In [1]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler

from sklearn.ensemble import RandomForestClassifier
from sklearn.neural_network import MLPClassifier

from sklearn.metrics import accuracy_score, f1_score

from statsmodels.stats.contingency_tables import mcnemar

In [2]:
df = pd.read_csv("../../data/raw/diabetes.csv")

X = df.drop("Outcome", axis=1)
y = df["Outcome"]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

In [3]:
scaler = MinMaxScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

In [4]:
rf = RandomForestClassifier(
    n_estimators=100,
    random_state=42
)

rf.fit(X_train, y_train)

pred_rf = rf.predict(X_test)

In [5]:
mlp = MLPClassifier(
    hidden_layer_sizes=(64,32),
    activation="relu",
    solver="adam",
    max_iter=500,
    random_state=42
)

mlp.fit(X_train_scaled, y_train)

pred_mlp = mlp.predict(X_test_scaled)

In [6]:
print("Random Forest Accuracy:", accuracy_score(y_test, pred_rf))
print("MLP Accuracy:", accuracy_score(y_test, pred_mlp))

print()

print("Random Forest F1:", f1_score(y_test, pred_rf))
print("MLP F1:", f1_score(y_test, pred_mlp))

Random Forest Accuracy: 0.7727272727272727
MLP Accuracy: 0.7337662337662337

Random Forest F1: 0.6464646464646465
MLP F1: 0.594059405940594


In [7]:
table = np.zeros((2,2), dtype=int)

for yt, rf_pred, ad_pred in zip(y_test, pred_rf, pred_mlp):

    rf_correct = (rf_pred == yt)
    ad_correct = (ad_pred == yt)

    if rf_correct and ad_correct:
        table[0,0] += 1

    elif rf_correct and not ad_correct:
        table[0,1] += 1

    elif not rf_correct and ad_correct:
        table[1,0] += 1

    else:
        table[1,1] += 1

print("Contingency Table")
print(table)

result = mcnemar(table, exact=True)

print("\nStatistic :", result.statistic)
print("P-value   :", result.pvalue)

if result.pvalue < 0.05:
    print("\nResult: Statistically Significant (p < 0.05)")
else:
    print("\nResult: Not Statistically Significant (p >= 0.05)")

Contingency Table
[[106  13]
 [  7  28]]

Statistic : 7.0
P-value   : 0.26317596435546875

Result: Not Statistically Significant (p >= 0.05)


In [8]:
results = pd.DataFrame({
    "Comparison": ["MLP vs Random Forest"],
    "P_value": [result.pvalue],
    "Significant": [result.pvalue < 0.05]
})

results.to_csv(
    "../../results/statistical_significance.csv",
    index=False
)

print(results)

             Comparison   P_value  Significant
0  MLP vs Random Forest  0.263176        False


### Statistical Significance Analysis

McNemar's test was used to compare the prediction performance of the Random Forest classifier and the baseline Multi-Layer Perceptron (MLP) on the same test set. The obtained p-value was 0.2632, which is greater than the significance threshold of 0.05. Therefore, the null hypothesis cannot be rejected, indicating that the observed performance difference between the two models is not statistically significant.